# Combine 3 Relation to Make 1 Unnormalized Dummy Relation 

In [3]:
import pandas as pd

# Load the three tables
customers = pd.read_csv("customers.csv")
orders    = pd.read_csv("orders.csv")
products  = pd.read_csv("products.csv")

# Clean up product price column (remove '?' or '₹' symbols) then force numeric
products["Price(Rs)"] = pd.to_numeric(
    products["Price(Rs)"].astype(str).str.replace(r"[^\d.]", "", regex=True),
    errors="coerce"
)

# Force quantity to numeric too
orders["quantity"] = pd.to_numeric(orders["quantity"], errors="coerce")

# Merge everything into one flat unnormalized table
merged = orders.merge(customers, on="customer_id", how="left")
merged = merged.merge(products,  on="product_id",  how="left")

# Add a derived total price column (extra denormalization)
merged["total_price(Rs)"] = merged["quantity"] * merged["Price(Rs)"]

# Reorder columns
merged = merged[[
    "Order_id", "date",
    "customer_id", "customer_name",
    "product_id", "product_name", "Price(Rs)",
    "quantity", "total_price(Rs)"
]]

merged.to_csv("unnormalized_orders.csv", index=False)
print(f"Done! {len(merged)} rows written to unnormalized_orders.csv")
print(merged.head())

Done! 310 rows written to unnormalized_orders.csv
   Order_id        date  customer_id      customer_name  product_id  \
0      1236  01/01/2024           74    Arjun, 40, Male       148.0   
1      1428  01/01/2024           54  Priya, 33, Female       149.0   
2      1585  01/01/2024           71     Liam, 42, Male       248.0   
3      1240  02/01/2024           56   Emma, 53, Female       242.0   
4      1311  03/01/2024           79    Ayaan, 32, Male       299.0   

      product_name  Price(Rs)  quantity  total_price(Rs)  
0        1. Apples      150.0      13.0           1950.0  
1       2. Bananas       40.0       1.0             40.0  
2       3. Oranges       80.0       3.0            240.0  
3        4. Grapes      200.0      15.0           3000.0  
4  5. Strawberries      300.0      18.0           5400.0  


C:\Users\Robben's Laptop\AppData\Local\Temp\ipykernel_25148\3469029089.py:19: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  merged = merged.merge(products,  on="product_id",  how="left")
